In [ ]:
# Load neccesary imports from library
from datasets import Dataset, load_dataset

# Load the training split of the dataset
ds = load_dataset(dataset_name, split="train")

# Filter for the first 500 samples of the dataset
filtered_ds = Dataset.from_dict(ds[:500])

In [ ]:
def create_intent_example(row):
    # Fill out the columns in the prompt
    row['intent_example'] = f"Query: {row['instruction']}\nIntent: {row['intent']}"
    return row

# Call the ds method to apply our preprocessing function to all rows
processed_dataset = dataset.map(create_intent_example)
# Print the intent_example in the first row of the processed data
print(processed_dataset[0]['intent_example'])

In [ ]:
from datasets import load_from_disk

# Save the dataset to disk
ds.save_to_disk("preprocessed_dataset")

# Load the dataset from disk
ds_preprocessed = load_from_disk("preprocessed_dataset")

# Print the first element of the loaded dataset
print(ds_preprocessed[0])

In [ ]:
config_dict = {
    # Define the model
    "model": {"_component_": "torchtune.models.llama3_2.llama3_2_1b"},
    # Define the batch size
    "batch_size": 8,
    # Define the device type
    "device": "cuda",
    "epochs": 15,
    "optimizer": {"_component_": "bitsandbytes.optim.PagedAdamW8bit", "lr": 3e-05},
    "dataset": {"_component_": "custom_dataset"},
    "output_dir": "/tmp/finetune_results"
}

In [ ]:
config_dict = {
    # Update the model
    "model": {"_component_": "torchtune.models.llama3_2.llama3_2_3b"},
    "batch_size": 8,
    "device": "cuda",
    "optimizer": {"_component_": "bitsandbytes.optim.PagedAdamW8bit", "lr": 3e-05},
    "dataset": {"_component_": "custom_dataset"},
    "output_dir": "/tmp/finetune_results"
}

# Save the updated configuration to a new YAML file
with open("custom_recipe.yaml", "w") as yaml_file:
    yaml.dump(config_dict, yaml_file)

In [ ]:
# Load helper class for the training arguments from the correct library
from transformers import TrainingArguments
training_arguments = TrainingArguments(
  	# Set learning rate
  	learning_rate=2e-3,
    warmup_ratio=0.03,
  	num_train_epochs=3,
    output_dir='/tmp',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    save_steps=10,
    logging_steps=2,
    lr_scheduler_type='constant',
    report_to='none'
)

In [ ]:
# Import the supervised fine-tuning class
from trl import SFTTrainer

# Instantiate fine-tuning class
trainer = SFTTrainer(
  	# Pass necessary arguments
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_arguments,
)

# Start training 
trainer.train()

In [ ]:
# Import the evaluation library from Hugging Face
import evaluate

# Instantiate your evaluate library and load the ROUGE metric
rouge_evaluator = evaluate.load('rouge')

# Fill in the method, and place your reference answers and test answers
results = rouge_evaluator.compute(predictions=test_answers, references=reference_answers)

# Extract the ROUGE1 score from the results dictionary
final_score = results['rouge1']
print(final_score)

In [ ]:
# Import LoRA configuration class
from peft import LoraConfig

# Instantiate LoRA configuration with values
lora_config = LoraConfig(
  	r=12,
    lora_alpha=8,
  	task_type="CAUSAL_LM",
    lora_dropout=0.05,
    bias="none",
    target_modules=['q_proj', 'v_proj']
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_arguments,
  	# Pass the lora_config to trainer
  	peft_config=lora_config,
)

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    # Set rank parameter 
  	r=2,
  	# Set scaling factor
    lora_alpha=4,
  	# Set the type of task
  	task_type="CAUSAL_LM",
    lora_dropout=0.05,
    bias="none",
    target_modules=['q_proj', 'v_proj']
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_arguments,
  	peft_config=peft_config,
)

trainer.train()

In [ ]:
# Import quantization configuration class
from transformers import BitsAndBytesConfig
# Instantiate quantization configuration
bnb_config = BitsAndBytesConfig(
    # Set 8-bit loading
    load_in_8bit=True,
)
model = AutoModelForCausalLM.from_pretrained(
    "Maykeye/TinyLLama-v0",
    # Set quantization parameters to load quantized model
    quantization_config=bnb_config,
    low_cpu_mem_usage=True
)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
  	# Set quantization type to normalized 4-bit
    bnb_4bit_quant_type="nf4",
  	# Set compute data type to be bfloat16
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    "Maykeye/TinyLLama-v0",
    quantization_config=bnb_config,
    low_cpu_mem_usage=True
)